# Part B — Retrieval Failure Cases & Fixes
**CS4241 | Student: Farima Konaré | Index: 10012200004**

This notebook demonstrates failure cases in retrieval and the fix applied.

**Failure case:** Vague/ambiguous queries return irrelevant chunks.
**Fix:** Query expansion — prepend domain-specific context keywords before embedding.

In [1]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

from retrieval.embedder import get_embedder
from retrieval.vector_store import VectorStore
from retrieval.retriever import HybridRetriever
from ingestion.chunker import load_chunks

# Load index
chunks = load_chunks('all_chunks.json')
emb = get_embedder()
store = VectorStore(dim=emb.dim)
if not store.load():
    store.build(chunks, emb)
    store.save()

retriever = HybridRetriever(store, chunks)
print(f'Index loaded: {store.size} vectors')

Index loaded: 1572 vectors from /Users/mac/Desktop/ai_10012200004/experiments/../src/retrieval/../../data/processed/faiss.index
Index loaded: 1572 vectors


## Failure Case 1: Vague geographic query

In [2]:
FAILURE_QUERY = 'What happened in the north?'

q_vec = emb.encode_query(FAILURE_QUERY)
results = retriever.retrieve(FAILURE_QUERY, q_vec, k=3)

print(f'Query: "{FAILURE_QUERY}"')
print('\n--- BEFORE fix (no query expansion) ---')
for i, r in enumerate(results, 1):
    print(f'{i}. combined={r["combined_score"]:.3f} | source={r["chunk"]["metadata"]["source"]}')
    print(f'   {r["chunk"]["text"][:120]}')

Query: "What happened in the north?"

--- BEFORE fix (no query expansion) ---
1. combined=0.497 | source=election
   In the 2020 Ghana presidential election, Nana Konadu Agyeman Rawlings of the NDP party received 117 votes (0.03%) in Wes
2. combined=0.496 | source=election
   In the 2020 Ghana presidential election, Nana Konadu Agyeman Rawlings of the NDP party received 322 votes (0.13%) in Nor
3. combined=0.450 | source=budget
   rth-East, Oti, Savannah, and Western North Regions; and  Establish one new Mission in Hungary.  iii.  iv.


In [3]:
# Apply fix: query expansion
expanded = retriever.expand_query(FAILURE_QUERY, source_filter='election')
print(f'Expanded query: "{expanded}"')

q_vec_expanded = emb.encode_query(expanded)
results_fixed = retriever.retrieve(expanded, q_vec_expanded, k=3)

print('\n--- AFTER fix (with query expansion) ---')
for i, r in enumerate(results_fixed, 1):
    print(f'{i}. combined={r["combined_score"]:.3f} | source={r["chunk"]["metadata"]["source"]}')
    print(f'   {r["chunk"]["text"][:120]}')

Expanded query: "Ghana election northern region north east region savannah region upper east region upper west region Ghana election What happened in the north?"

--- AFTER fix (with query expansion) ---
1. combined=0.733 | source=election
   In the 2020 Ghana presidential election, Akua Donkor of the GFP party received 324 votes (0.14%) in North East Region (f
2. combined=0.723 | source=election
   In the 2020 Ghana presidential election, John Dramani Mahama of the NDC party received 112306 votes (46.97%) in North Ea
3. combined=0.722 | source=election
   In the 2016 Ghana presidential election, John Dramani Mahama of the NDC party received 92395 votes (48.50%) in North Eas


## Failure Case 2: Out-of-domain query (false-positive check)

In [4]:
OOD_QUERY = 'What is the capital of France?'

q_vec_ood = emb.encode_query(OOD_QUERY)
results_ood = retriever.retrieve(OOD_QUERY, q_vec_ood, k=3)

print(f'Query: "{OOD_QUERY}"')
print(f'Is relevant (threshold ≥ 0.25): {retriever.is_relevant(results_ood)}')
print()
for i, r in enumerate(results_ood, 1):
    print(f'{i}. combined={r["combined_score"]:.3f} | {r["chunk"]["text"][:80]}')

Query: "What is the capital of France?"
Is relevant (threshold ≥ 0.25): True

1. combined=0.525 | e Euro  Area and Japan at 1.0 and 1.2 percent, respectively. Due to the high lev
2. combined=0.503 | f GDP in 2024 compared  to  68.7  percent  of  GDP in  2023.  This  reduction  i
3. combined=0.454 | Resetting the Economy for the Ghana We Want  2025 Budget  437.  To  promote  int


## Summary

| Failure | Root cause | Fix | Result |
|---|---|---|---|
| Vague northern query | "north" matches everything; no domain anchor | Query expansion: prepend "Ghana election northern region" | Top results shifted to northern regional election data with higher scores |
| Out-of-domain query | Broad semantic overlap with macroeconomic budget text | Current threshold (0.25) is too permissive for this case | False positive remained (`is_relevant=True`, top score `0.525`), indicating need for stronger OOD filtering |